# Create References On Works (oxjob #2347)

Each work's reference list as `GET /works/{id}/references` will serve it: one row per work, `references` an ordered list of
`{raw_reference_string, raw_doi, work_id}` (Casey + Jason 17 Nov, Casey 20 Nov 2026). One rule: the best
version's longest list in printed order; then each work matched only on another of the work's lists, once, with that
list's text; then every other served citation (legacy MAG, frozen arrays, curated adds) with null raw fields. Linked ids are
exactly the work's served rows in `work_citations`, so they equal `referenced_works` (up to its 5,000 cap); both raw fields
null marks a citation with no printed reference. A line with no printed text and no DOI shows what its DOI field holds instead
(a URL), else its deposited fields (author. year. title), as #2187 does at ingest for entries with no title. Built in its own daily job after End 2 End (Casey: keep pressure off the pipeline).
Plan and schema: oxjobs `working/references-on-works/`.

In [ ]:
CREATE OR REPLACE TABLE openalex.works._references_on_works_lines
CLUSTER BY (work_id)
TBLPROPERTIES ('delta.deletedFileRetentionDuration' = '7 days', 'delta.logRetentionDuration' = '7 days')
AS
WITH m AS (SELECT id AS k, merge_into_id AS v FROM openalex.works.merged_work_ids_map),
lists AS (
  SELECT provenance, native_id_namespace, native_id,
    CASE WHEN version_rank <= 2 THEN 1 WHEN version_rank = 3 THEN 2 ELSE 3 END AS version_tier
  FROM openalex.works.work_reference_lists_current
),
ln AS (
  SELECT coalesce(mc.v, wr.citing_work_id) AS work_id, wr.provenance, wr.native_id_namespace, wr.native_id, wr.ref_ind,
    coalesce(l.version_tier, 4) AS version_tier,
    CASE wr.provenance WHEN 'crossref' THEN 1 WHEN 'pubmed' THEN 2 WHEN 'repo' THEN 3 WHEN 'datacite' THEN 4 ELSE 5 END AS source_rank,
    nullif(regexp_extract(trim(wr.doi), r'(?i)^(?:https?://(?:dx\.)?doi\.org/|doi:\s*)?(10\..+)$', 1), '') AS doi,
    nullif(trim(wr.raw), '') AS raw, nullif(trim(wr.doi), '') AS doi_field, nullif(concat_ws('. ',
      nullif(trim(TRAILING '.' FROM trim(wr.authors)), ''), nullif(trim(wr.year), ''),
      nullif(trim(TRAILING '.' FROM trim(wr.title)), '')), '') AS fields_text,
    coalesce(md.v, wr.cited_work_id) AS matched_work_id
  FROM openalex.works.work_references wr
  LEFT JOIN lists l USING (provenance, native_id_namespace, native_id)
  LEFT JOIN m mc ON mc.k = wr.citing_work_id
  LEFT JOIN m md ON md.k = wr.cited_work_id
),
ln2 AS (
  SELECT work_id, provenance, native_id_namespace, native_id, ref_ind, version_tier, source_rank, doi, matched_work_id,
    coalesce(raw, CASE WHEN doi IS NULL THEN coalesce(doi_field, fields_text) END) AS raw_reference_string
  FROM ln
),
ranked AS (
  SELECT *, dense_rank() OVER (PARTITION BY work_id
      ORDER BY version_tier, list_lines DESC, source_rank, provenance, native_id_namespace, native_id) AS list_rank
  FROM (SELECT *, count(*) OVER (PARTITION BY work_id, provenance, native_id_namespace, native_id) AS list_lines FROM ln2)
),
served AS (SELECT citing_work_id, cited_work_id FROM openalex.works.work_citations WHERE served)
SELECT r.work_id, r.list_rank, r.provenance, r.native_id_namespace, r.native_id, r.ref_ind, r.raw_reference_string,
  'https://doi.org/' || r.doi AS raw_doi,
  CASE WHEN s.cited_work_id IS NOT NULL THEN r.matched_work_id END AS linked_work_id
FROM ranked r
LEFT JOIN served s ON s.citing_work_id = r.work_id AND s.cited_work_id = r.matched_work_id
LEFT SEMI JOIN openalex.works.openalex_works_base w ON w.id = r.work_id
WHERE TRUE

In [ ]:
CREATE OR REPLACE TABLE openalex.works.references_on_works
CLUSTER BY (work_id)
TBLPROPERTIES ('delta.deletedFileRetentionDuration' = '7 days', 'delta.logRetentionDuration' = '7 days')
AS
WITH l AS (SELECT * FROM openalex.works._references_on_works_lines),
b1 AS (
  SELECT work_id, 1 AS blk, CAST(ref_ind AS BIGINT) AS k1, 0 AS k2, raw_reference_string, raw_doi, linked_work_id
  FROM l WHERE list_rank = 1
),
shown1 AS (SELECT DISTINCT work_id, linked_work_id FROM b1 WHERE linked_work_id IS NOT NULL),
b2 AS (
  SELECT l.work_id, 2 AS blk, CAST(l.list_rank AS BIGINT) AS k1, l.ref_ind AS k2, l.raw_reference_string, l.raw_doi,
    l.linked_work_id
  FROM l LEFT ANTI JOIN shown1 s ON s.work_id = l.work_id AND s.linked_work_id = l.linked_work_id
  WHERE l.list_rank > 1 AND l.linked_work_id IS NOT NULL
  QUALIFY row_number() OVER (PARTITION BY l.work_id, l.linked_work_id ORDER BY l.list_rank, l.ref_ind) = 1
),
shown AS (SELECT work_id, linked_work_id FROM shown1 UNION ALL SELECT work_id, linked_work_id FROM b2),
b3 AS (
  SELECT c.citing_work_id AS work_id, 3 AS blk, c.cited_work_id AS k1, 0 AS k2, CAST(NULL AS STRING) AS raw_reference_string,
    CAST(NULL AS STRING) AS raw_doi, c.cited_work_id AS linked_work_id
  FROM openalex.works.work_citations c
  LEFT ANTI JOIN shown s ON s.work_id = c.citing_work_id AND s.linked_work_id = c.cited_work_id
  WHERE c.served
),
u AS (SELECT * FROM b1 UNION ALL SELECT * FROM b2 UNION ALL SELECT * FROM b3)
SELECT work_id,
  transform(array_sort(collect_list(struct(blk, k1, k2, raw_reference_string, raw_doi, linked_work_id))),
    x -> named_struct('raw_reference_string', x.raw_reference_string, 'raw_doi', x.raw_doi, 'work_id', x.linked_work_id)) AS `references`,
  count(*) AS lines, count(linked_work_id) AS linked_lines, count_if(blk = 1) AS printed_lines,
  count_if(blk = 2) AS other_list_lines, count_if(blk = 3) AS unprinted_citations, current_timestamp() AS built_at
FROM u
GROUP BY work_id

In [ ]:
-- Each work's linked ids must be exactly its served citations, compared as rows (array set operations are unreliable
-- in big queries); fails the run otherwise.
WITH shown AS (
  SELECT DISTINCT r.work_id, x.work_id AS cited_work_id
  FROM openalex.works.references_on_works r LATERAL VIEW explode(r.`references`) e AS x
  WHERE x.work_id IS NOT NULL
),
served AS (SELECT citing_work_id AS work_id, cited_work_id FROM openalex.works.work_citations WHERE served),
c AS (
  SELECT count(*) AS pairs, count_if(served.work_id IS NULL) AS shown_not_served, count_if(shown.work_id IS NULL) AS served_not_shown
  FROM shown FULL OUTER JOIN served USING (work_id, cited_work_id)
)
SELECT *, CASE WHEN shown_not_served + served_not_shown > 0
  THEN raise_error(concat('references_on_works: ', shown_not_served, ' linked ids not served, ', served_not_shown,
                          ' served citations missing'))
  END AS failed
FROM c